# Egyptian Legal Assistant

An Arabic retrieval-augmented generation (RAG) prototype using Egyptian legal texts, multilingual embeddings, FAISS, Groq and Gradio.

**Educational prototype, not legal advice.** Retrieved text and generated citations must be checked against current legislation. This cleaned edition has not been tested end to end.

Run the cells in order in Google Colab. GPU is recommended for embeddings. Set `GROQ_API_KEY` and `GROQ_MODEL` in environment variables or Colab Secrets. The original Groq model IDs are retired, so choose an available model from your own Groq account. Groq usage may incur charges. Do not enter confidential case details.

## 1. Install dependencies

In [ ]:
%pip install -q langchain-community langchain-huggingface langchain-groq langchain-text-splitters sentence-transformers faiss-cpu beautifulsoup4 gradio torch

## 2. Imports

In [ ]:
import os
from functools import partial

import torch
import gradio as gr
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import WebBaseLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

## 3. Private configuration

In Colab, open the key icon, add `GROQ_API_KEY` and `GROQ_MODEL`, and enable notebook access. Never paste a key into a cell. An optional `HUGGINGFACEHUB_API_TOKEN` secret is supported if model access requires it. Public model downloads may not need it.

Both generation model IDs from the original notebook (`qwen-qwq-32b` and `deepseek-r1-distill-llama-70b`) have been shut down. See the [Groq deprecation notice](https://console.groq.com/docs/deprecations). This edition does not claim a replacement model was evaluated.

In [ ]:
def read_setting(name, required=False):
    value = os.environ.get(name, "").strip()
    if not value:
        try:
            from google.colab import userdata
            value = (userdata.get(name) or "").strip()
        except ImportError:
            pass
        except Exception:
            # Do not expose secret-service details in notebook output.
            pass
    if required and not value:
        raise ValueError(f"Set {name} in environment variables or Colab Secrets.")
    return value

api_key = read_setting("GROQ_API_KEY", required=True)
model_id = read_setting("GROQ_MODEL", required=True)
if model_id in {"qwen-qwq-32b", "deepseek-r1-distill-llama-70b"}:
    raise ValueError("The selected model is retired. Choose an available Groq model.")

hf_token = read_setting("HUGGINGFACEHUB_API_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

llm = ChatGroq(model=model_id, api_key=api_key)

## 4. Arabic embeddings

The original notebook uses normalized embeddings from `sayed0am/arabic-english-bge-m3`. CUDA is used when available; CPU fallback is slower.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding = HuggingFaceEmbeddings(
    model_name="sayed0am/arabic-english-bge-m3",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

## 5. Load and split source pages

The source URLs are preserved from the original notebook. Web pages may contain navigation, outdated text or missing amendments. This prototype does not isolate article text or verify legal currency.

In [ ]:
# Egyptian Laws URLs from manshurat.org
urls = [
    "https://manshurat.org/node/14675",  # الدستور المصري
    "https://manshurat.org/node/14677",  # قانون العقوبات المصري
]

In [ ]:
loader = WebBaseLoader(urls)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)
splits_l = text_splitter.split_documents(documents)
print(f"Document loaded and split into {len(splits_l)} chunks")

## 6. Build a local FAISS index

Retrieve five chunks per query. The saved index is a generated local artifact, not part of the repository.

In [ ]:
# Create FAISS vector store
vectorstore = FAISS.from_documents(splits_l, embedding)

# Optional: Save to disk
vectorstore.save_local("faiss_index")


# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

## 7. Arabic answer prompt

The prompt asks for Arabic answers, relevant legal articles and an explicit response when context is insufficient. These are instructions, not guarantees of correct answers.

In [ ]:
template = """
       التعليمات:
      بصفتك مساعدًا قانونيًا مهذبًا ومحترفًا، مهمتك هي الإجابة على استفسارات المستخدم بشكل فعال وباللغة العربية فقط.

      يرجى الالتزام بالشروط التالية:
      (0) قراءة السؤال بعناية وفهمه تمامًا
      (1) الرد حصريًا باللغة العربية الفصحى
      (2) البدء بتأكيد فهم السؤال بصيغة: "بخصوص سؤالك عن [موضوع السؤال]..."
      (3) تقديم إجابة قانونية دقيقة مدعومة بالنصوص التشريعية
      (4) إذا كانت المعلومات غير متوفرة، قل: "لا تتوفر معلومات كافية عن هذا الموضوع في مصادرنا الحالية"
      (5) الاستشهاد بالمواد القانونية ذات الصلة
      (6) مراجعة الإجابة للتأكد من:
          - السلامة اللغوية (عربي فقط)
          - الدقة القانونية
          - الوضوح والترتيب المنطقي

      تحذير: ممنوع استخدام أي كلمات أو جمل غير عربية.

       الاستفسار:
       السؤال: {question}

       السياق: {context}

       الإجابة:
       إجابة مفصلة مع المراجع:
"""

In [ ]:
prompt = PromptTemplate.from_template(template)

In [ ]:
# Define a function to format documents for context
def format_docs(docs):
    return "\n\n".join(f"Document {i+1} (Source: {doc.metadata.get('source', 'Unknown')}):\n{doc.page_content}"
                       for i, doc in enumerate(docs))

In [ ]:
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 8. Optional smoke test

Uncomment the query to make one Groq request. This is not an accuracy benchmark.

In [ ]:
# result = chain.invoke("ما هي عقوبة السرقة في القانون المصري؟")
# print(result)

## 9. Arabic demo interface

Run this cell to launch the demo in the notebook. Sharing is off by default. Setting `share=True` creates an externally accessible temporary link; do not do that with confidential information. Each question sends retrieved context and the question to Groq.

In [ ]:
def get_answer(question, chain):
    try:
        if not question.strip():
            return "⚠️ الرجاء إدخال سؤال صحيح"

        response = chain.invoke(question)
        return str(getattr(response, 'content', response))

    except Exception:
        return "تعذر إنشاء الإجابة. تحقق من إعدادات النموذج والاتصال ثم حاول مجددًا."

def create_interface(chain):
    with gr.Blocks(
        theme=gr.themes.Soft(),
        css="""
        .gradio-container {direction: rtl;}
        .suggestions {background: #f0f8ff; padding: 15px; border-radius: 10px; margin: 10px 0;}
        .suggest-title {font-weight: bold; color: #2e86c1;}
        .suggest-btn {margin: 5px; background: #eaf2f8 !important;}
        """
    ) as demo:

        gr.Markdown("""
        <div style='text-align: center'>
            <h1>(المتر)👨⚖️ المساعد القانوني المصري</h1>
            <p>نموذج تجريبي للأسئلة عن النصوص المتاحة، وليس بديلًا عن محامٍ.</p>
        </div>
        """)

        # تعريف متغير السؤال أولاً
        question = gr.Textbox(
            label="السؤال القانوني",
            placeholder="اكتب سؤالك هنا أو اختر من الاقتراحات أعلاه",
            lines=3
        )

        # قسم الرسائل الاقتراحية (يجب أن يأتي بعد تعريف question)

        submit_btn = gr.Button("الحصول على الإجابة", variant="primary")
        answer = gr.Textbox(label="الإجابة", lines=10, interactive=False)

        submit_btn.click(
            partial(get_answer, chain=chain),
            inputs=question,
            outputs=answer
        )

        # أمثلة إضافية
        gr.Examples(
            examples=[
                ["ما هي حقوق الموظف عند الفصل التعسفي؟"],
                ["كيف يتم إثبات عقد الزواج العرفي؟"],
                ["ما هي مدة التقادم في الدعاوى الجنائية؟"],
                ["ما هي شروط الحصول على الجنسية المصرية؟"],
                ["كيفية التبليغ عن جريمة إلكترونية؟"]
            ],
            inputs=question,
            label="المزيد من الأسئلة الشائعة"
        )

    return demo



demo = create_interface(chain)
demo.launch(share=False, debug=False)